# Figure 5 noise and threshold playground

Use this notebook to regenerate the MoS2, SrTiO3, and rattled graphene edge tests over different background/noise/count settings, then tune the dynamic threshold objective visually.

The default objective is conservative: maximize `F0.5`, which penalizes false positives more strongly than missed atoms. Change `OBJECTIVE` or the threshold grid below if you want to explore a different tradeoff.

In [1]:
from __future__ import annotations

import csv
import json
from dataclasses import replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

# Resolve the repo root whether the notebook is run from notebooks/ or the repo root.
CWD = Path.cwd().resolve()
REPO = CWD if (CWD / 'scripts' / 'make_manuscript_figures.py').exists() else CWD.parent
assert (REPO / 'scripts' / 'make_manuscript_figures.py').exists(), f'Could not find repo root from {CWD}'

import sys
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts import make_manuscript_figures as mf

print(REPO)

D:\GitHub\BlobNet


## Experiment controls

Edit these values and rerun the cells below. `COUNT_SCALES` controls the Poisson count scale. `BACKGROUND_PROFILES_TO_RUN` chooses which background profiles are included in a sweep.

In [2]:
SHAPE = (512, 512)
SIGMA_RANGE = (2.6, 3.2)
SEED = 41

COUNT_SCALES = [2, 4, 8, 16, 32, 64]
BACKGROUND_PROFILES_TO_RUN = ['quiet', 'low', 'medium', 'high']
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.05), 2).tolist()

# Options: 'f0.5' is conservative, 'f1' is balanced, 'recall_at_precision' asks for high recall subject to MIN_PRECISION.
OBJECTIVE = 'f0.5'
MIN_PRECISION = 0.985

OUTPUT_DIR = REPO / 'outputs' / 'figure5_notebook_playground'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILTERS = [32, 64, 128, 256]
DROPOUT = 0.2
DEVICE = mf._device_from_name('auto')
print('device:', DEVICE)
print('output:', OUTPUT_DIR)

device: cuda
output: D:\GitHub\BlobNet\outputs\figure5_notebook_playground


In [3]:
BACKGROUND_PROFILES = {
    'quiet': {
        'background_range': (0.006, 0.030),
        'gradient_range': (-0.012, 0.012),
        'inhomogeneous_background_range': (0.006, 0.025),
        'low_frequency_noise_range': (0.004, 0.020),
        'read_noise_std_range': (0.006, 0.020),
    },
    'low': {
        'background_range': (0.010, 0.055),
        'gradient_range': (-0.020, 0.020),
        'inhomogeneous_background_range': (0.012, 0.045),
        'low_frequency_noise_range': (0.010, 0.045),
        'read_noise_std_range': (0.010, 0.035),
    },
    'medium': {
        'background_range': (0.018, 0.090),
        'gradient_range': (-0.035, 0.035),
        'inhomogeneous_background_range': (0.025, 0.075),
        'low_frequency_noise_range': (0.018, 0.075),
        'read_noise_std_range': (0.018, 0.060),
    },
    'high': {
        'background_range': (0.040, 0.180),
        'gradient_range': (-0.060, 0.060),
        'inhomogeneous_background_range': (0.050, 0.130),
        'low_frequency_noise_range': (0.040, 0.140),
        'read_noise_std_range': (0.050, 0.150),
    },
}

MODELS = [
    mf.ModelSpec('square', 'Square model', REPO / 'outputs' / 'manuscript_models' / 'square' / 'unet_best.pth'),
    mf.ModelSpec('hexagonal', 'Hexagonal model', REPO / 'outputs' / 'manuscript_models' / 'hexagonal' / 'unet_best.pth'),
    mf.ModelSpec('random', 'Measured random model', REPO / 'outputs' / 'manuscript_models' / 'random_dense' / 'unet_best.pth'),
]

for spec in MODELS:
    print(spec.key, spec.checkpoint, spec.checkpoint.exists())

square D:\GitHub\BlobNet\outputs\manuscript_models\square\unet_best.pth True
hexagonal D:\GitHub\BlobNet\outputs\manuscript_models\hexagonal\unet_best.pth True
random D:\GitHub\BlobNet\outputs\manuscript_models\random_dense\unet_best.pth True


## Load models

This takes a moment on the first run. Reuse the loaded models for the rest of the notebook.

In [4]:
loaded_models = {
    spec.key: mf._load_blobnet_model(spec.checkpoint, DEVICE, MODEL_FILTERS, DROPOUT)
    for spec in MODELS
}
print('loaded:', ', '.join(loaded_models))

loaded: square, hexagonal, random


## Helpers

These helpers keep the Figure 5 formulation unchanged while letting the notebook override only the noise/background/count parameters.

In [5]:
def edge_config(count_scale: float, profile_name: str):
    base = mf._edge_figure_render_config(
        SHAPE,
        SIGMA_RANGE,
        total_counts_range=(float(count_scale), float(count_scale)),
        quiet_background=True,
    )
    return replace(base, **BACKGROUND_PROFILES[profile_name])


def make_edge_record(case_key: str, profile_name: str, count_scale: float):
    cfg = edge_config(count_scale, profile_name)
    original = mf._edge_figure_render_config

    def patched_edge_config(shape_arg, sigma_range_arg, total_counts_range=(35.0, 250.0), quiet_background=False):
        if tuple(shape_arg) == SHAPE and tuple(float(v) for v in sigma_range_arg) == SIGMA_RANGE:
            return cfg
        return original(shape_arg, sigma_range_arg, total_counts_range=total_counts_range, quiet_background=quiet_background)

    mf._edge_figure_render_config = patched_edge_config
    try:
        if case_key == 'mos2_edge':
            return mf._make_mos2_edge_record(SHAPE, SEED, SIGMA_RANGE, total_counts_range=(count_scale, count_scale), quiet_background=True)
        if case_key == 'srtio3_edge':
            return mf._make_sto_edge_record(SHAPE, SEED + 101, SIGMA_RANGE, total_counts_range=(count_scale, count_scale), quiet_background=True)
        if case_key == 'graphene_rattled_edge':
            return mf._make_graphene_rattled_edge_record(SHAPE, SEED + 202, SIGMA_RANGE, total_counts_range=(count_scale, count_scale), quiet_background=True)
        raise ValueError(case_key)
    finally:
        mf._edge_figure_render_config = original


def metric_scores(classes: dict[str, Any]) -> dict[str, float]:
    tp = float(classes['tp'])
    fp = float(classes['fp'])
    fn = float(classes['fn'])
    precision = tp / max(tp + fp, 1.0)
    recall = tp / max(tp + fn, 1.0)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-8)
    beta = 0.5
    f05 = (1.0 + beta * beta) * precision * recall / max(beta * beta * precision + recall, 1e-8)
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f0.5': f05,
        'false_discovery_rate': fp / max(tp + fp, 1.0),
    }


def choose_threshold(prediction: np.ndarray, coordinates: np.ndarray, objective: str = OBJECTIVE):
    args = SimpleNamespace(peak_min_distance=3, peak_window_size=5, localization_match_distance=3.0)
    candidates = []
    for threshold in THRESHOLD_GRID:
        classes = mf._localization_classes_for_figure(
            prediction,
            coordinates,
            threshold_rel=float(threshold),
            min_distance=args.peak_min_distance,
            peak_window_size=args.peak_window_size,
            match_distance=args.localization_match_distance,
        )
        scores = metric_scores(classes)
        candidates.append((float(threshold), classes, scores))

    if objective == 'f1':
        key = lambda item: (item[2]['f1'], -int(item[1]['fp']), item[2]['precision'], -int(item[1]['fn']), item[0])
    elif objective == 'recall_at_precision':
        feasible = [item for item in candidates if item[2]['precision'] >= MIN_PRECISION]
        pool = feasible if feasible else candidates
        key = lambda item: (item[2]['precision'] >= MIN_PRECISION, item[2]['recall'], -int(item[1]['fp']), item[2]['precision'], item[0])
        return max(pool, key=key)
    else:
        key = lambda item: (item[2]['f0.5'], -int(item[1]['fp']), item[2]['precision'], -int(item[1]['fn']), item[0])
    return max(candidates, key=key)


def predict_case(record: dict[str, Any]):
    image = np.asarray(record['image'], dtype=np.float32)
    return {spec.key: mf._predict_array(loaded_models[spec.key], image, DEVICE) for spec in MODELS}

## Render one setting

Start here when you are tuning. Change `profile_name`, `count_scale`, or `objective` and rerun this cell.

In [8]:
def render_figure5_setting(profile_name='medium', count_scale=16, objective=OBJECTIVE, save=True):
    cases = [
        ('mos2_edge', make_edge_record('mos2_edge', profile_name, count_scale)),
        ('srtio3_edge', make_edge_record('srtio3_edge', profile_name, count_scale)),
        ('graphene_rattled_edge', make_edge_record('graphene_rattled_edge', profile_name, count_scale)),
    ]
    fig = plt.figure(figsize=(20.0, 11.4), constrained_layout=True)
    grid = fig.add_gridspec(len(cases), 5, width_ratios=[1.05, 1.0, 1.0, 1.0, 1.0], wspace=0.05, hspace=0.05)
    rows = []

    for row_index, (case_key, record) in enumerate(cases):
        image = np.asarray(record['image'], dtype=np.float32)
        target = np.asarray(record['target'], dtype=np.float32)
        coordinates = np.asarray(record['coordinates'], dtype=np.float32)

        mf._plot_clean_image(fig.add_subplot(grid[row_index, 0]), image, '')
        mf._plot_ground_truth_for_figure(fig.add_subplot(grid[row_index, 1]), target)
        predictions = predict_case(record)

        for col_index, spec in enumerate(MODELS, start=2):
            threshold, classes, scores = choose_threshold(predictions[spec.key], coordinates, objective)
            ax = fig.add_subplot(grid[row_index, col_index])
            mf._plot_localization_scatter_for_figure(
                ax,
                classes,
                SHAPE,
                spec.label,
                show_legend=True,
                marker_color=mf.MODEL_COLORS.get(spec.key, '#2e7d32'),
            )
            rows.append({
                'background_profile': profile_name,
                'poisson_count_scale': count_scale,
                'case': case_key,
                'model': spec.key,
                'threshold_rel': threshold,
                'tp': int(classes['tp']),
                'fp': int(classes['fp']),
                'fn': int(classes['fn']),
                **scores,
            })

    out_path = None
    if save:
        out_path = FIGURE_DIR / f'figure5_{objective.replace(".", "_")}_{profile_name}_counts_{count_scale}.png'
        fig.savefig(out_path, dpi=220, bbox_inches='tight')
        print(out_path)
    plt.show()
    return pd.DataFrame(rows), out_path


one_setting_metrics, one_setting_path = render_figure5_setting('medium', 16, OBJECTIVE)
one_setting_metrics

D:\GitHub\BlobNet\outputs\figure5_notebook_playground\figures\figure5_f0_5_medium_counts_16.png


C:\Users\ahoust17\AppData\Local\Temp\ipykernel_77492\1541792812.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,background_profile,poisson_count_scale,case,model,threshold_rel,tp,fp,fn,precision,recall,f1,f0.5,false_discovery_rate
0,medium,16,mos2_edge,square,0.25,463,0,10,1.000000,0.978858,0.989316,0.995699,0.000000
1,medium,16,mos2_edge,hexagonal,0.50,473,0,0,1.000000,1.000000,1.000000,1.000000,0.000000
2,medium,16,mos2_edge,random,0.85,460,5,13,0.989247,0.972516,0.980810,0.985855,0.010753
3,medium,16,srtio3_edge,square,0.35,680,0,2,1.000000,0.997067,0.998532,0.999412,0.000000
4,medium,16,srtio3_edge,hexagonal,0.20,641,1,41,0.998442,0.939883,0.968278,0.986154,0.001558
5,medium,16,srtio3_edge,random,0.85,667,1,15,0.998503,0.978006,0.988148,0.994335,0.001497
6,medium,16,graphene_rattled_edge,square,0.25,519,1,0,0.998077,1.000000,0.999038,0.998461,0.001923
7,medium,16,graphene_rattled_edge,hexagonal,0.80,517,1,2,0.998069,0.996146,0.997107,0.997684,0.001931
8,medium,16,graphene_rattled_edge,random,0.90,515,0,4,1.000000,0.992293,0.996132,0.998449,0.000000


## Run a manifold sweep

This can take several minutes because it runs all three models for every case/background/count combination.

In [ ]:
def run_sweep(objective=OBJECTIVE, profiles=BACKGROUND_PROFILES_TO_RUN, counts=COUNT_SCALES):
    all_rows = []
    image_paths = []
    for profile_name in profiles:
        for count_scale in counts:
            print(f'rendering {profile_name}, counts={count_scale}, objective={objective}')
            metrics, path = render_figure5_setting(profile_name, count_scale, objective, save=True)
            all_rows.append(metrics)
            image_paths.append(path)
            plt.close('all')
    metrics = pd.concat(all_rows, ignore_index=True)
    csv_path = OUTPUT_DIR / f'figure5_{objective.replace(".", "_")}_metrics.csv'
    metrics.to_csv(csv_path, index=False)
    manifest = {
        'objective': objective,
        'min_precision': MIN_PRECISION,
        'threshold_grid': THRESHOLD_GRID,
        'count_scales': list(counts),
        'background_profiles': {key: BACKGROUND_PROFILES[key] for key in profiles},
        'figures': [str(path) for path in image_paths],
    }
    manifest_path = OUTPUT_DIR / f'figure5_{objective.replace(".", "_")}_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(csv_path)
    print(manifest_path)
    return metrics, image_paths


# Uncomment to regenerate the full sweep from inside the notebook.
# sweep_metrics, sweep_paths = run_sweep(OBJECTIVE)

## Inspect saved conservative sweep

This reads the sweep generated outside the notebook. It is useful as a quick baseline before you rerun anything.

In [ ]:
CONSERVATIVE_SWEEP = REPO / 'outputs' / 'figure5_fp_conservative_noise_sweep'
conservative_csv = CONSERVATIVE_SWEEP / 'figure5_conservative_threshold_metrics.csv'
if conservative_csv.exists():
    conservative_metrics = pd.read_csv(conservative_csv)
    summary = (
        conservative_metrics
        .groupby(['background_profile', 'poisson_count_scale'], as_index=False)
        .agg(tp=('tp', 'sum'), fp=('fp', 'sum'), fn=('fn', 'sum'), mean_f05=('f0_5', 'mean'), mean_f1=('f1', 'mean'), precision=('precision', 'mean'), recall=('recall', 'mean'))
        .sort_values(['mean_f05', 'mean_f1'], ascending=False)
    )
    display(summary)
else:
    print('No conservative sweep found yet:', conservative_csv)

In [ ]:
if conservative_csv.exists():
    for profile_name in ['quiet', 'low', 'medium', 'high']:
        path = CONSERVATIVE_SWEEP / f'contact_{profile_name}.png'
        if path.exists():
            print(path)
            display(Image(filename=str(path)))

## Threshold curves for one model/case

This cell shows why a given threshold was selected. It is useful when a panel still looks like it has too many FPs or too many FNs.

In [ ]:
def threshold_curve(case_key='mos2_edge', model_key='random', profile_name='medium', count_scale=64):
    record = make_edge_record(case_key, profile_name, count_scale)
    prediction = mf._predict_array(loaded_models[model_key], np.asarray(record['image'], dtype=np.float32), DEVICE)
    coordinates = np.asarray(record['coordinates'], dtype=np.float32)
    rows = []
    for threshold in THRESHOLD_GRID:
        classes = mf._localization_classes_for_figure(prediction, coordinates, threshold, 3, 5, 3.0)
        rows.append({'threshold_rel': threshold, 'tp': classes['tp'], 'fp': classes['fp'], 'fn': classes['fn'], **metric_scores(classes)})
    curve = pd.DataFrame(rows)
    best_threshold, _, best_scores = choose_threshold(prediction, coordinates, OBJECTIVE)
    ax = curve.plot(x='threshold_rel', y=['precision', 'recall', 'f1', 'f0.5'], marker='o', figsize=(8, 4))
    ax.axvline(best_threshold, color='black', linestyle='--', linewidth=1)
    ax.set_ylim(0, 1.02)
    ax.set_title(f'{case_key} / {model_key} / {profile_name} / counts {count_scale}')
    plt.show()
    display(curve)
    print('selected threshold:', best_threshold, best_scores)
    return curve


curve = threshold_curve('mos2_edge', 'random', 'medium', 64)